<div style="padding: 20px; background: linear-gradient(90deg, #1D976C 0%, #93F9B9 100%); border-radius: 10px; color: black;">
    <h1 style="color: black; border-bottom: none;">🔧 Module 7.3: Corrective RAG (CRAG)</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">Using LangGraph to grade, self-correct, and fallback to Web Search.</p>
</div>

---

## 1. What is CRAG?

Standard RAG is passive: it pulls documents and blindly feeds them to the LLM, even if they are completely irrelevant.
**CRAG** adds an active **Grading Phase** and a **Fallback Mechanism**.

1. Retrieve documents from the Vector DB.
2. An LLM **Grades** the documents: *"Are these actually relevant to the question?"*
3. If GOOD → Proceed to generate answer.
4. If POOR → Fall back to **Web Search**!

## 2. Enter LangGraph
Because this requires complex loops and conditional routing, we cannot use standard LangChain sequential chains. We must build a **State Machine** using LangGraph.

In [1]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from dotenv import load_dotenv
import os

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
load_dotenv()

# 1. Define the Global State of our Machine
class CRAGState(TypedDict):
    query: str
    documents: List[Document]
    grade: str
    final_answer: str
    rewritten_query: str

docs = [
    Document(page_content="Python is an interpreted, high-level programming language."),
    Document(page_content="LangChain is a framework for building applications powered by language models."),
    Document(page_content="Vector databases store and retrieve embeddings efficiently."),
]
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vs = Chroma.from_documents(docs, embeddings, collection_name="crag_demo")
print("Database loaded with Python/LangChain facts.")

E:\001_Github_Repo_all\Advanced-RAG-Systems\.venv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Database loaded with Python/LangChain facts.


## 3. Building the LangGraph Nodes
Each function represents a "Node" in our flowchart.

In [2]:
groq_api_key = os.environ.get("GROQ_API_KEY")

if groq_api_key:
    llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
    
    # --- NODE 1: Retrieve ---
    def retrieve(state: CRAGState) -> CRAGState:
        print("[NODE] Retrieving from Vector DB...")
        docs = vs.similarity_search(state["query"], k=2)
        return {**state, "documents": docs}
        
    # --- NODE 2: Grade ---
    grade_prompt = ChatPromptTemplate.from_template("""
    Is the following document relevant to the user query?
    Answer EXACTLY 'yes' or 'no'. Do not explain.
    
    Query: {query}
    Document: {document}
    """)
    
    def grade_documents(state: CRAGState) -> CRAGState:
        print("[NODE] Grading documents...")
        grades = []
        chain = grade_prompt | llm | StrOutputParser()
        for doc in state["documents"]:
            result = chain.invoke({"query": state["query"], "document": doc.page_content})
            grades.append(result.strip().lower())
            
        overall = "relevant" if "yes" in grades else "irrelevant"
        print(f"  → Grade: {overall.upper()}")
        return {**state, "grade": overall}
        
    # --- NODE 3: Fallback Rewrite ---
    def rewrite_query(state: CRAGState) -> CRAGState:
        print("[NODE] Rewriting query for web search...")
        rewrite = (
            ChatPromptTemplate.from_template("Rewrite this query for better web search: {query}")
            | llm | StrOutputParser()
        ).invoke({"query": state["query"]})
        return {**state, "rewritten_query": rewrite.strip()}
        
    # --- NODE 4: Fallback Web Search ---
    def web_search(state: CRAGState) -> CRAGState:
        print("[NODE] Performing Web Search...")
        q = state.get("rewritten_query", state["query"])
        # Simulated web search
        sim = [Document(page_content=f"[Web Result] The Steam Engine was invented by James Watt and Thomas Newcomen.")]
        return {**state, "documents": sim}
        
    # --- NODE 5: Generate ---
    gen_prompt = ChatPromptTemplate.from_template("""
    Answer the question using ONLY the context below.
    Context: {context}
    Question: {question}
    """)
    
    def generate(state: CRAGState) -> CRAGState:
        print("[NODE] Generating final answer...")
        context = "\n".join(d.page_content for d in state["documents"])
        answer = (gen_prompt | llm | StrOutputParser()).invoke({
            "context": context, "question": state["query"]
        })
        return {**state, "final_answer": answer}
        
    # --- ROUTER ---
    def route_grade(state: CRAGState) -> str:
        return "generate" if state["grade"] == "relevant" else "rewrite_query"
else:
    print("GROQ_API_KEY missing.")

## 4. Assembling and Running the Graph

In [3]:
if groq_api_key:
    # Build the State Machine Flowchart
    graph = StateGraph(CRAGState)
    graph.add_node("retrieve", retrieve)
    graph.add_node("grade", grade_documents)
    graph.add_node("rewrite_query", rewrite_query)
    graph.add_node("web_search", web_search)
    graph.add_node("generate", generate)
    
    graph.set_entry_point("retrieve")
    graph.add_edge("retrieve", "grade")
    graph.add_conditional_edges("grade", route_grade, {"generate": "generate", "rewrite_query": "rewrite_query"})
    graph.add_edge("rewrite_query", "web_search")
    graph.add_edge("web_search", "generate")
    graph.add_edge("generate", END)
    
    crag = graph.compile()
    
    # --- RUN 1: Query is in the database ---
    print("\n=== TEST 1 (In DB) ===")
    result = crag.invoke({"query": "What is LangChain?", "documents": [], "grade": "", "final_answer": "", "rewritten_query": ""})
    print(f"\nFINAL ANSWER: {result['final_answer']}")
    
    # --- RUN 2: Query is NOT in the database (Triggers Web Fallback) ---
    print("\n=== TEST 2 (Not in DB) ===")
    result = crag.invoke({"query": "Who invented the steam engine?", "documents": [], "grade": "", "final_answer": "", "rewritten_query": ""})
    print(f"\nFINAL ANSWER: {result['final_answer']}")


=== TEST 1 (In DB) ===
[NODE] Retrieving from Vector DB...
[NODE] Grading documents...


  → Grade: RELEVANT
[NODE] Generating final answer...

FINAL ANSWER: LangChain is a framework for building applications powered by language models.

=== TEST 2 (Not in DB) ===
[NODE] Retrieving from Vector DB...
[NODE] Grading documents...


  → Grade: IRRELEVANT
[NODE] Rewriting query for web search...


[NODE] Performing Web Search...
[NODE] Generating final answer...

FINAL ANSWER: The Steam Engine was invented by James Watt and Thomas Newcomen.
